In [ ]:
# Step 1 — Start Hadoop and Hive
$HADOOP_HOME/sbin/start-dfs.sh
$HADOOP_HOME/sbin/start-yarn.sh
hive

# Step 2 — Prepare Local Data Files
mkdir -p ~/hive_data

cat > ~/hive_data/products.csv << EOF
1,Mobile,Electronics,15000
2,Laptop,Electronics,55000
3,Shirt,Clothing,1200
4,Shoes,Clothing,2500
EOF

cat > ~/hive_data/products_electronics.csv << EOF
1,Mobile,Electronics,15000
2,Laptop,Electronics,55000
EOF

cat > ~/hive_data/products_clothing.csv << EOF
3,Shirt,Clothing,1200
4,Shoes,Clothing,2500
EOF

# Step 3 — Enter Hive Shell
hive

CREATE DATABASE IF NOT EXISTS productdb;
USE productdb;

CREATE TABLE products(
    product_id INT,
    product_name STRING,
    category STRING,
    price FLOAT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE;

LOAD DATA LOCAL INPATH '/home/'${USER}'/hive_data/products.csv' INTO TABLE products;

SELECT * FROM products;

CREATE TABLE products_partitioned(
    product_id INT,
    product_name STRING,
    price FLOAT
)
PARTITIONED BY (category STRING)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE;

exit;

hdfs dfs -mkdir -p /user/${USER}/productdb/products_partitioned/category=Electronics
hdfs dfs -mkdir -p /user/${USER}/productdb/products_partitioned/category=Clothing

hdfs dfs -put ~/hive_data/products_electronics.csv /user/${USER}/productdb/products_partitioned/category=Electronics/
hdfs dfs -put ~/hive_data/products_clothing.csv /user/${USER}/productdb/products_partitioned/category=Clothing/

hive

USE productdb;

ALTER TABLE products_partitioned ADD PARTITION (category='Electronics')
LOCATION '/user/'${USER}'/productdb/products_partitioned/category=Electronics';

ALTER TABLE products_partitioned ADD PARTITION (category='Clothing')
LOCATION '/user/'${USER}'/productdb/products_partitioned/category=Clothing';

SHOW PARTITIONS products_partitioned;

SELECT * FROM products_partitioned WHERE category='Electronics';

SELECT product_name, price, price * 1.1 AS price_with_tax FROM products;

SELECT UPPER(product_name) AS upper_name, ROUND(price, 0) AS rounded_price FROM products;

CREATE VIEW expensive_products AS
SELECT * FROM products WHERE price > 5000;

SELECT * FROM expensive_products;

CREATE INDEX idx_price ON TABLE products (price)
AS 'COMPACT'
WITH DEFERRED REBUILD;

ALTER INDEX idx_price ON products REBUILD;

INSERT OVERWRITE DIRECTORY '/user/'${USER}'/hive_output/products'
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
SELECT * FROM products;

exit;

hdfs dfs -cat /user/${USER}/hive_output/products/*
